In [2]:
import pandas as pd

In [3]:
df=pd.read_csv("D:\project\houseprice\Cleaned_data_for_model.csv")

<>:1: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:1: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
C:\Users\yousef\AppData\Local\Temp\ipykernel_3148\3660519083.py:1: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  df=pd.read_csv("D:\project\houseprice\Cleaned_data_for_model.csv")


In [4]:
df=df.drop_duplicates()

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99499 entries, 0 to 99498
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0     99499 non-null  int64  
 1   property_type  99499 non-null  str    
 2   price          99499 non-null  int64  
 3   location       99499 non-null  str    
 4   city           99499 non-null  str    
 5   baths          99499 non-null  int64  
 6   purpose        99499 non-null  str    
 7   bedrooms       99499 non-null  int64  
 8   Area_in_Marla  99499 non-null  float64
dtypes: float64(1), int64(4), str(4)
memory usage: 6.8 MB


In [6]:
df=df.drop("Unnamed: 0",axis=1)

In [7]:
df.isnull().sum()

property_type    0
price            0
location         0
city             0
baths            0
purpose          0
bedrooms         0
Area_in_Marla    0
dtype: int64

In [8]:
df["property_type"].value_counts()

property_type
House            58169
Flat             26658
Upper Portion     8539
Lower Portion     5549
Penthouse          255
Room               241
Farm House          88
Name: count, dtype: int64

In [9]:
df["bedrooms"].value_counts()

bedrooms
3    34888
2    23245
4    17458
5    14355
6     6275
1     2984
0      294
Name: count, dtype: int64

In [10]:
df["location"].value_counts()

location
DHA Defence               11787
Bahria Town Karachi        6697
Bahria Town Rawalpindi     5257
Bahria Town                4437
Gulistan-e-Jauhar          3532
                          ...  
Damba                         1
Sultan Colony                 1
Defence Fort                  1
Sihala Valley                 1
Shahra-e-Liaquat              1
Name: count, Length: 1389, dtype: int64

In [11]:
x=df.drop("price",axis=1)
y=df["price"]

In [12]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [13]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((79599, 7), (19900, 7), (79599,), (19900,))

In [14]:
cat_cols=x.select_dtypes(include=["object","str"]).columns
num_cols=x.select_dtypes(include=["float64","int64"]).columns

In [15]:
cat_cols

Index(['property_type', 'location', 'city', 'purpose'], dtype='str')

In [16]:
num_cols

Index(['baths', 'bedrooms', 'Area_in_Marla'], dtype='str')

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_pipeline = Pipeline(steps=[
    ("scale", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_pipeline, cat_cols),
        ("num", num_pipeline, num_cols)
    ]
)

In [27]:
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score
models={
    "RF":RandomForestRegressor(random_state=42),
    "Gb":GradientBoostingRegressor(random_state=42),
    "XG":XGBRegressor(random_state=42),
    
}
results={}
for name,model in models.items():
    pipeline=Pipeline(steps=[
        ("preprocessor",preprocessor),
        ("model",model)
    ])
    pipeline.fit(x_train,y_train)
    y_pred=pipeline.predict(x_test)
    scores=r2_score(y_test,y_pred)
    results[name]=pipeline
    print(f"{name} score is {scores}")

RF score is 0.915963189507886
Gb score is 0.8463950656256289
XG score is 0.9034453630447388


In [20]:
from sklearn.model_selection import cross_val_score
for name,model in models.items():
    pipeline=Pipeline(steps=[
        ("preprocessor",preprocessor),
        ("model",model)
    ])
    scores=cross_val_score(pipeline,x,y,cv=5,scoring="r2")
    print(f"{name} Cv score{scores.mean():.4f}")

RF Cv score0.9108
Gb Cv score0.8428
XG Cv score0.9013


In [25]:
print(models["RF"])
print(type(models["RF"]))

RandomForestRegressor(random_state=42)
<class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [42]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    "model__n_estimators": randint(100, 300),
    "model__max_depth": [10, 20, 30]
}

rf = results["RF"]

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=10,  
    cv=3,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

random_search.fit(x_train, y_train)

print("Best params:", random_search.best_params_)
print("Best score:", random_search.best_score_)

Best params: {'model__max_depth': 30, 'model__n_estimators': 279}
Best score: 0.9115311281967612


In [43]:
best_model=random_search.best_estimator_

In [47]:
import joblib
from sklearn.metrics import r2_score
y_pred=best_model.predict(x_test)
print(r2_score(y_test,y_pred))

0.9128737035986793


In [48]:
import joblib 
joblib.dump(best_model, "house_model.pkl")

['house_model.pkl']

In [49]:
from sklearn.metrics import mean_squared_error,mean_absolute_error
print("mse",mean_squared_error(y_test,y_pred))
print("mae",mean_absolute_error(y_test,y_pred))

mse 9972533560615.455
mae 1723321.7353746858
